In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kethaka_sap_cast import config

d = pd.read_csv(config.PROCESSED / "dataset.csv", parse_dates=["date"]).set_index("date")
FIG = config.REPORTS / "figures"
FIG.mkdir(parents=True, exist_ok=True)
d.head()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(d.index, d["total_l"], lw=0.8, alpha=0.5, label="daily")
ax.plot(d.index, d["total_l"].rolling(7, center=True).mean(), lw=2, label="7-day average")
ax.axvline(pd.Timestamp("2025-04-01"), color="grey", ls="--", lw=1)
ax.set(title="Estate sap collected per day, 2025", ylabel="litres")
ax.legend()
fig.savefig(FIG / "01_daily_sap.png", dpi=150, bbox_inches="tight")

In [ ]:
m = d.resample("MS").agg({"total_l": "mean", "precipitation_sum": "sum"})

fig, ax1 = plt.subplots(figsize=(10, 4))
ax1.bar(m.index, m["precipitation_sum"], width=20, color="lightsteelblue", label="rain (mm/month)")
ax1.set_ylabel("rain, mm per month")
ax2 = ax1.twinx()
ax2.plot(m.index, m["total_l"], marker="o", color="darkgreen", label="sap (L/day)")
ax2.set_ylabel("average sap, litres per day")
ax1.set_title("Monthly rainfall and average daily sap, 2025")
fig.savefig(FIG / "02_monthly_sap_rain.png", dpi=150, bbox_inches="tight")

In [ ]:
windows = range(1, 31)
corr = [d["total_l"].corr(d["precipitation_sum"].rolling(n).sum().shift(1)) for n in windows]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(list(windows), corr, marker="o")
ax.axhline(0, color="grey", lw=1)
ax.set(title="Correlation of daily sap with rain over the previous N days",
       xlabel="N days of rain before", ylabel="correlation")
fig.savefig(FIG / "03_rain_lag_correlation.png", dpi=150, bbox_inches="tight")

In [ ]:
change = d["total_l"].diff().abs()
period = np.where(d.index < "2025-04-01", "Jan-Mar", "Apr-Dec")
print(change.groupby(period).describe().round(2))

fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot([change[period == "Jan-Mar"].dropna(), change[period == "Apr-Dec"].dropna()],
           tick_labels=["Jan-Mar", "Apr-Dec"])
ax.set(title="Day-to-day change in estate sap", ylabel="litres")
fig.savefig(FIG / "04_daily_change_by_period.png", dpi=150, bbox_inches="tight")

In [ ]:
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
d.groupby(d.index.day_name())["total_l"].mean().reindex(order).round(1)